# Notebook 06 — Arquitectura VLA con Transformer

Este notebook construye y comprueba una arquitectura Vision-Language-Action (VLA) sin realizar todavía el entrenamiento completo. El flujo es **CLIP → Transformer de fusión → decoder de acciones**. Se reutilizan los embeddings y las acciones generados en los notebooks 03 y 04.

## 1. Objetivos y configuración

Se definirán los parámetros, la semilla y los módulos propios. Para que la comprobación sea rápida, el notebook trabaja directamente con el caché de CLIP; el encoder completo también queda disponible en `VLA` para posteriores usos con imágenes y tokens.

In [1]:
from pathlib import Path
import json
import os
import random
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

ROOT_DIR = Path(os.getcwd()).resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.action_decoder import ActionDecoder
from src.train import action_loss
from src.transformer import MultimodalTransformer
from src.vla_model import VLA

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(os.cpu_count() or 1)

EMBEDDING_DIM = 512
FUSION_DIM = 256
DECODER_HIDDEN_DIM = 128
OUTPUT_DIM = 8
BATCH_SIZE = 8
print(f'Dispositivo: {DEVICE} | semilla: {SEED}')
print(f'Configuración: {EMBEDDING_DIM} → {FUSION_DIM} → {DECODER_HIDDEN_DIM} → {OUTPUT_DIM}')
# Rutas compartidas: no dependen del directorio desde el que se abra el notebook.
from src.project_config import PROJECT_DIR, DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, PROCESSED_DIR, CACHE_DIR as SHARED_CACHE_DIR, taco_play_dir, ensure_project_dirs, experiment_dirs
ROOT_DIR = PROJECT_DIR
ensure_project_dirs()


Dispositivo: cpu | semilla: 42
Configuración: 512 → 256 → 128 → 8


## 2. Carga de embeddings y acciones

Cada archivo `.npz` contiene las mismas muestras y particiones que 04: embeddings visuales, embeddings textuales y acciones normalizadas. Se comprueban las dimensiones, la correspondencia fila a fila y que no existan valores inválidos.

In [2]:
CACHE_DIR = ROOT_DIR / 'data' / 'cache_embeddings'
PARAMS_PATH = ROOT_DIR / 'data' / 'parametros_normalizacion.json'

def cargar_particion(nombre):
    ruta = CACHE_DIR / f'{nombre}.npz'
    if not ruta.exists():
        raise FileNotFoundError(f'No existe {ruta}. Ejecuta antes 04_clip_embeddings.ipynb.')
    with np.load(ruta) as datos:
        visual = np.asarray(datos['imagenes'], dtype=np.float32)
        textual = np.asarray(datos['textos'], dtype=np.float32)
        acciones = np.asarray(datos['acciones'], dtype=np.float32)
    if not (visual.ndim == textual.ndim == acciones.ndim == 2):
        raise ValueError(f'{nombre}: todos los arrays deben ser bidimensionales')
    if not (len(visual) == len(textual) == len(acciones)):
        raise ValueError(f'{nombre}: embeddings y acciones no tienen el mismo número de muestras')
    if visual.shape[1:] != (EMBEDDING_DIM,) or textual.shape[1:] != (EMBEDDING_DIM,):
        raise ValueError(f'{nombre}: se esperaban embeddings de 512 dimensiones')
    if acciones.shape[1:] != (OUTPUT_DIM,):
        raise ValueError(f'{nombre}: se esperaban acciones de 8 dimensiones')
    if not (np.isfinite(visual).all() and np.isfinite(textual).all() and np.isfinite(acciones).all()):
        raise ValueError(f'{nombre}: hay valores no finitos')
    return visual, textual, acciones

particiones = {nombre: cargar_particion(nombre) for nombre in ('train', 'validation', 'test')}
for nombre, (visual, textual, acciones) in particiones.items():
    print(f'{nombre:10s}: visual={visual.shape}, texto={textual.shape}, acciones={acciones.shape}')
    assert np.logical_and(acciones >= 0, acciones <= 1).all(), f'{nombre}: acciones fuera de [0, 1]'

# El JSON confirma que la normalización compartida por 03 y 04 está disponible.
assert PARAMS_PATH.exists(), f'No existe {PARAMS_PATH}'
with PARAMS_PATH.open(encoding='utf-8') as archivo:
    parametros_normalizacion = json.load(archivo)
assert len(parametros_normalizacion['minimo']) == OUTPUT_DIM
print('Correspondencia de muestras, particiones y orden de acciones comprobada.')

train     : visual=(190212, 512), texto=(190212, 512), acciones=(190212, 8)
validation: visual=(23760, 512), texto=(23760, 512), acciones=(23760, 8)
test      : visual=(23826, 512), texto=(23826, 512), acciones=(23826, 8)
Correspondencia de muestras, particiones y orden de acciones comprobada.


## 3. Preparación de los lotes

Se mantienen separadas las particiones train, validation y test. Solo se usa un lote pequeño de train para las comprobaciones de esta etapa. El orden de la acción es `[terminate, Δx, Δy, Δz, Δrx, Δry, Δrz, gripper]`.

In [3]:
datasets = {
    nombre: TensorDataset(*(torch.from_numpy(array) for array in datos))
    for nombre, datos in particiones.items()
}
generator = torch.Generator().manual_seed(SEED)
loaders = {
    'train': DataLoader(datasets['train'], batch_size=BATCH_SIZE, shuffle=True, generator=generator),
    'validation': DataLoader(datasets['validation'], batch_size=BATCH_SIZE, shuffle=False),
    'test': DataLoader(datasets['test'], batch_size=BATCH_SIZE, shuffle=False),
}
image_batch, text_batch, target_batch = next(iter(loaders['train']))
print(f'Lote: imágenes={tuple(image_batch.shape)}, texto={tuple(text_batch.shape)}, acciones={tuple(target_batch.shape)}')
assert target_batch.shape[-1] == OUTPUT_DIM
assert torch.all((target_batch >= 0) & (target_batch <= 1))

Lote: imágenes=(8, 512), texto=(8, 512), acciones=(8, 8)


## 4. Transformer de fusión multimodal

Los embeddings de imagen y texto se convierten en dos tokens de 256 dimensiones. Las posiciones aprendibles conservan la identidad de cada token y el encoder aplica atención entre ambas modalidades; finalmente se hace mean pooling.

In [4]:
fusion = MultimodalTransformer(
    embedding_dim=EMBEDDING_DIM, hidden_dim=FUSION_DIM,
    num_layers=2, num_heads=4, feedforward_dim=512, dropout=0.1
).to(DEVICE)
with torch.no_grad():
    fused_batch = fusion(image_batch.to(DEVICE), text_batch.to(DEVICE))
print(f'Salida del Transformer: {tuple(fused_batch.shape)}')
assert fused_batch.shape == (len(image_batch), FUSION_DIM)
assert torch.isfinite(fused_batch).all()
print('Atención multimodal comprobada correctamente.')

Salida del Transformer: (8, 256)
Atención multimodal comprobada correctamente.


## 5. Decodificador de acciones

El decoder devuelve ocho salidas normalizadas: MSE para las seis físicas y probabilidades para `terminate` y pinza. El criterio común aplica BCE a ambas variables binarias.

In [5]:
decoder = ActionDecoder(FUSION_DIM, DECODER_HIDDEN_DIM, OUTPUT_DIM).to(DEVICE)
with torch.no_grad():
    decoded_batch = decoder(fused_batch)
print(f'Salida del decoder: {tuple(decoded_batch.shape)}')
assert decoded_batch.shape == (len(image_batch), OUTPUT_DIM)
assert torch.isfinite(decoded_batch).all()
assert torch.all((decoded_batch >= 0) & (decoded_batch <= 1))
print('Las ocho componentes tienen dimensiones y rango válidos.')

Salida del decoder: (8, 8)
Las ocho componentes tienen dimensiones y rango válidos.


## 6. Arquitectura VLA completa

Se integran el Transformer y el decoder en `VLA`. Al recibir embeddings precalculados no es necesario volver a cargar vídeos ni CLIP, lo que permite comprobar la arquitectura de forma ligera.

In [6]:
model = VLA(clip_encoder=None, embedding_dim=EMBEDDING_DIM, fusion_dim=FUSION_DIM, decoder_hidden_dim=DECODER_HIDDEN_DIM).to(DEVICE)
model.eval()
with torch.no_grad():
    predictions = model(image_embeddings=image_batch.to(DEVICE), text_embeddings=text_batch.to(DEVICE))
print(f'Predicción VLA: {tuple(predictions.shape)}')
assert predictions.shape == (len(image_batch), OUTPUT_DIM)
assert torch.isfinite(predictions).all()
assert torch.all((predictions >= 0) & (predictions <= 1))

# La misma interfaz funciona con distintos tamaños de lote.
for tamano in (1, 2, min(16, len(particiones['train'][0]))):
    visual = torch.from_numpy(particiones['train'][0][:tamano]).to(DEVICE)
    textual = torch.from_numpy(particiones['train'][1][:tamano]).to(DEVICE)
    salida = model(image_embeddings=visual, text_embeddings=textual)
    assert salida.shape == (tamano, OUTPUT_DIM)
print('Flujo completo comprobado para varios tamaños de lote.')

Predicción VLA: (8, 8)
Flujo completo comprobado para varios tamaños de lote.


## 7. Pérdida, gradientes y congelación de CLIP

La pérdida común es MSE para `x..rz` más BCE para `terminate` y, tras la comprobación del notebook 02, para la pinza binaria.

In [7]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)
predictions = model(image_embeddings=image_batch.to(DEVICE), text_embeddings=text_batch.to(DEVICE))
loss = action_loss(predictions, target_batch.to(DEVICE))
assert torch.isfinite(loss)
loss.backward()

gradientes = [param.grad for param in model.parameters() if param.requires_grad]
assert gradientes and all(gradiente is not None for gradiente in gradientes)
assert all(torch.isfinite(gradiente).all() for gradiente in gradientes)
optimizer.step()

parametros_entrenables = sum(param.numel() for param in model.parameters() if param.requires_grad)
parametros_totales = sum(param.numel() for param in model.parameters())
print(f'Pérdida inicial: {loss.item():.6f}')
print(f'Gradientes válidos: {len(gradientes)} tensores')
print(f'Parámetros: {parametros_totales:,} totales | {parametros_entrenables:,} entrenables')
assert model.clip_encoder is None  # equivale a trabajar con CLIP congelado y cacheado

Pérdida inicial: 0.787639
Gradientes válidos: 35 tensores
Parámetros: 1,351,816 totales | 1,351,816 entrenables


## 8. Resumen y preparación para el notebook 07

La arquitectura está lista para entrenarse con los lotes del caché. El notebook 07 podrá reutilizar `VLA`, `action_loss` y las tres particiones sin modificar el orden de las acciones ni reconstruir CLIP.

In [8]:
print('Resumen de componentes:')
print('  CLIP: embeddings cacheados de 512 dimensiones, encoder congelado')
print('  Fusión: 2 tokens → Transformer Encoder (2 capas, 4 cabezas, d_model=256) → mean pooling')
print('  Decoder: MLP 256 → 128 → 8 con ReLU y Sigmoid')
print('  Acción: [terminate, Δx, Δy, Δz, Δrx, Δry, Δrz, gripper]')
print(f'  Parámetros totales: {parametros_totales:,}')
print(f'  Parámetros entrenables: {parametros_entrenables:,}')
print('Notebook 06 completado correctamente; modelo preparado para 07.')

Resumen de componentes:
  CLIP: embeddings cacheados de 512 dimensiones, encoder congelado
  Fusión: 2 tokens → Transformer Encoder (2 capas, 4 cabezas, d_model=256) → mean pooling
  Decoder: MLP 256 → 128 → 8 con ReLU y Sigmoid
  Acción: [terminate, Δx, Δy, Δz, Δrx, Δry, Δrz, gripper]
  Parámetros totales: 1,351,816
  Parámetros entrenables: 1,351,816
Notebook 06 completado correctamente; modelo preparado para 07.


## Resumen de resultados

Este notebook utiliza la configuración centralizada, crea las carpetas de salida necesarias y deja los artefactos del experimento en una carpeta propia.